In [1]:
#Importing libraries
import os
import json
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY")
)

## Agent Without Memory

In [2]:
def ask_agent(question):
    response = client.responses.create(
        model="gpt-4.1-mini",
        input=question
    )

    return response.output_text

In [3]:
print(ask_agent("My name is Alex and I am learning Python."))

Hi Alex! That’s great to hear you’re learning Python. How can I assist you with your Python journey today? Do you have any questions or something specific you want to work on?


In [4]:
print(ask_agent("What is my name and what am I learning?"))

I’m sorry, but I don’t have any information about your name or what you are learning based on our current conversation. Could you please tell me more?


## Short Term Conversation Memory

In [5]:
conversation_history = []

def chat_with_memory(user_message):

    conversation_history.append({
        "role": "user",
        "content": user_message
    })

    response = client.responses.create(
        model="gpt-4.1-mini",
        input=conversation_history
    )

    assistant_message = response.output_text

    conversation_history.append({
        "role": "assistant",
        "content": assistant_message
    })

    return assistant_message


In [6]:
print(
    chat_with_memory(
        "My name is Alex and I am learning Python."
    )
)

Hi Alex! That's great to hear that you're learning Python. How can I assist you with your Python learning today? Do you have any questions or need help with a specific topic?


In [8]:
print(
    chat_with_memory(
        "What is my name and what am I learning?"
    )
) #This is application-managed conversation history, not a model permanently learning or remembering the user. The history exists in your Python process and disappears when that process is restarted unless you save it.

Your name is Alex, and you are learning Python.


In [9]:
#Inspecting the conversation history
print(
    json.dumps(
        conversation_history,
        indent=2
    )
)

[
  {
    "role": "user",
    "content": "My name is Alex and I am learning Python."
  },
  {
    "role": "assistant",
    "content": "Hi Alex! That's great to hear that you're learning Python. How can I assist you with your Python learning today? Do you have any questions or need help with a specific topic?"
  },
  {
    "role": "user",
    "content": "What is my name and what am I learning?"
  },
  {
    "role": "assistant",
    "content": "Your name is Alex, and you are learning Python."
  },
  {
    "role": "user",
    "content": "What is my name and what am I learning?"
  },
  {
    "role": "assistant",
    "content": "Your name is Alex, and you are learning Python."
  }
]


## Long Term Memory using a JSON file

In [10]:
memory = {
    "name": "Alex",
    "learning": "Python",
    "experience_level": "beginner"
}

In [11]:
MEMORY_FILE = Path("agent_memory.json")

In [12]:
#Creating a function to save a memory
def save_memory(memory):
    with open(MEMORY_FILE, "w") as file:
        json.dump(memory, file, indent=4)

    print("Memory saved successfully.")

In [13]:
#Function to load memory
def load_memory():
    if MEMORY_FILE.exists():
        with open(MEMORY_FILE, "r") as file:
            return json.load(file)

    return {}

In [14]:
#TEst
user_memory = {
    "name": "Alex",
    "learning": "Python",
    "experience_level": "beginner"
}

save_memory(user_memory)

Memory saved successfully.


In [15]:
loaded_memory = load_memory()

print(loaded_memory)

{'name': 'Alex', 'learning': 'Python', 'experience_level': 'beginner'}


In [16]:
#Retrieve memory when answering questions
def ask_agent_with_long_term_memory(question):

    memory = load_memory()

    instructions = f"""
    You are a helpful AI learning assistant.

    Here is the saved information about the user:

    {json.dumps(memory, indent=2)}

    Use this information when it is relevant.
    Do not invent personal details.
    If the information is not available, say so.
    """

    response = client.responses.create(
        model="gpt-4.1-mini",
        instructions=instructions,
        input=question
    )

    return response.output_text

In [17]:
print(
    ask_agent_with_long_term_memory(
        "What am I learning, and what is my experience level?"
    )
)

You are learning Python, and your experience level is beginner.


In [18]:
#Now restart your notebook kernel, run the imports and function definitions again, and test the same question. The saved JSON file should still be available, provided you're running from the same working directory.

## Combining both Short Term and Long Term memory

In [19]:
def memory_agent(user_message):

    memory = load_memory()

    conversation_history.append({
        "role": "user",
        "content": user_message
    })

    instructions = f"""
    You are a helpful AI assistant.

    Saved user information:
    {json.dumps(memory, indent=2)}

    Use the saved information and conversation history
    when relevant. Do not invent missing facts.
    """

    response = client.responses.create(
        model="gpt-4.1-mini",
        instructions=instructions,
        input=conversation_history
    )

    answer = response.output_text

    conversation_history.append({
        "role": "assistant",
        "content": answer
    })

    return answer

In [20]:
print(
    memory_agent(
        "Can you suggest a beginner-friendly Python exercise?"
    )
)

Certainly, Alex! Here's a beginner-friendly Python exercise for you:

**Exercise: Write a Python program to check if a number is even or odd.**

**Instructions:**
1. Ask the user to input a number.
2. Convert the input to an integer.
3. Use the modulus operator `%` to determine if the number is even or odd.
4. Print a message saying whether the number is even or odd.

**Example:**
```
Enter a number: 7
7 is an odd number.
```

Would you like me to help you with the solution as well?
